# Classes: Dunder Methods, Inheritance, Composition, Registries

## What you'll practice

Build classes that feel native: containers that support `len`, iteration, indexing and slicing; base classes with template methods and `super()`;
composition with operator overloading; plugin registration via `__init_subclass__`; and finally a mini immutable table — your first taste of
DataFrame design.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/functions_oop/test_05_04_classes_inheritance_composition.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/05_functions_and_oop/04_classes_inheritance_composition.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · A native-feeling Batch container

**Implement:** `class Batch`

`Batch(records=None, batch_id="b0")` wraps a list. Support `len(b)`, `for r in b`, `b[0]`, `b[1:3]` (**returns a new `Batch`** with the same id),
`bool(b)` (False when empty), `append(record)`, `==` (same id and records), and `repr` → `Batch(id='b0', size=3)`. The constructor copies the input list.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_batch` in `tests/functions_oop/test_05_04_classes_inheritance_composition.py`.

**Check:** `pytest tests/functions_oop/test_05_04_classes_inheritance_composition.py -k test_batch`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Template method with inheritance and super()

**Implement:** `class FileSource`, `class CsvSource(FileSource)`, `class JsonlSource(FileSource)`

`FileSource(path)` provides `records()` — a generator that opens the file, skips blank lines, calls `self.parse(line_no, line)` for each, and yields dicts
(skipping lines where `parse` returns `None`); it counts `self.skipped`. Base `parse` raises `NotImplementedError`.
`CsvSource(path, delimiter=",")` calls `super().__init__(path)`, treats the first non-blank line as the header, and returns `None` for rows with the wrong number of fields.
`JsonlSource` returns `None` for malformed JSON or non-objects.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_csv_source, test_pipe_delimiter, test_jsonl_source` in `tests/functions_oop/test_05_04_classes_inheritance_composition.py`.

**Check:** `pytest tests/functions_oop/test_05_04_classes_inheritance_composition.py -k "test_csv_source or test_pipe_delimiter or test_jsonl_source or test_base_is_abstract_ish"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Composable pipelines with operators

**Implement:** `class Pipeline`

`Pipeline(*steps, name="pipeline")` holds steps; each step is a function `records -> records` (takes and returns an iterable).
`add(step)` appends and returns `self` (chaining). `run(records)` applies the steps in order and returns a **list**. `p1 | p2` returns a **new** pipeline
with p1's steps then p2's (name `"p1+p2"`); `p | func` appends a function. `len(p)` = number of steps; `repr` → `Pipeline('clean', steps=2)`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_pipeline_run_and_chain, test_pipeline_operators` in `tests/functions_oop/test_05_04_classes_inheritance_composition.py`.

**Check:** `pytest tests/functions_oop/test_05_04_classes_inheritance_composition.py -k "test_pipeline_run_and_chain or test_pipeline_operators"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Plugin registry with __init_subclass__

**Implement:** `class Transform`

Base class `Transform` keeps a class-level `registry: dict[str, type]`. Any subclass defining a class attribute `name = "..."` is registered
automatically (use `__init_subclass__`); duplicate names → `ValueError`; subclasses without `name` aren't registered (abstract helpers).
`Transform.create(name, **params)` instantiates by name (unknown → `KeyError` listing names). Instances are callables: `__call__(record) -> record`
delegates to `apply`. Then implement subclasses `Rename(mapping)` (`name="rename"`), `Cast(column, to)` (`name="cast"`, `to` is `"int"|"float"|"str"`),
`Drop(columns)` (`name="drop"`, a list of column names), and `run_transforms(record, specs)` where specs are `[{"name": ..., **params}, ...]` — config-driven transforms like Glue/dbt.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
run_transforms({'quantity': None}, [{'name': 'cast', 'column': 'quantity', 'to': 'float'}])
# -> {'quantity': None}
```

**Check:** `pytest tests/functions_oop/test_05_04_classes_inheritance_composition.py -k "test_registry_and_create or test_duplicate_and_abstract_helper or test_run_transforms"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · A mini immutable Table (DataFrame preview) · 🧩 Mini-project

**Implement:** `class Table`

Build `Table(rows: list[dict])` — immutable; every operation returns a **new** Table:
- `columns` property (union of keys, first-seen order); `len(t)`; iteration yields row dicts (copies); `t["col"]` → list of values (`None` if missing)
- `where(predicate)`, `select(*cols)`, `order_by(col, desc=False)` (`None` values always last), `with_column(name, func)` (func receives the row)
- `group_by(*keys).agg(**aggs)` where each agg is `(column, function)`, e.g. `agg(total=("amount", sum), n=("order_id", len))` → Table with key columns + agg columns, rows ordered by first appearance of each group
- `join(other, on, how="inner"|"left")` on a single key column (left join fills missing right columns with `None`; right-side duplicate keys produce multiple rows)
- `__repr__` → first 5 rows as an aligned text table + `[n rows x m columns]`
Then use it to answer: revenue per country from `datasets/csv/orders.csv`, top 3.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_table_basics_and_immutability, test_table_group_by_agg, test_table_join` in `tests/functions_oop/test_05_04_classes_inheritance_composition.py`.

**Check:** `pytest tests/functions_oop/test_05_04_classes_inheritance_composition.py -k "test_table_basics_and_immutability or test_table_group_by_agg or test_table_join or test_table_repr"`

**YOUR CODE ↓**